# Notebook 03: Location Scoring Model (for UMKM & Government)

## Objective
Build a robust machine learning pipeline to predict **skor_potensi** (location potential score, continuous 0-100) for UMKM locations across Jawa Barat. This model serves two purposes:
1. **UMKM Entrepreneurs**: Evaluate potential locations before starting or expanding a business
2. **Government**: Identify priority areas for intervention and resource allocation

## Methodology
- Multiple regression models compared (Linear, Random Forest, XGBoost, LightGBM, Gradient Boosting)
- Rigorous train/test split with stratification
- Hyperparameter tuning via RandomizedSearchCV
- SHAP-based model interpretation for actionable insights
- Government priority area identification with specific recommendations

## Section 1: Load & Prepare Data

We load the engineered dataset from Notebook 02 and prepare feature matrix X and target vector y.
Features exclude target variables (skor_potensi, is_survived_3yr), geographic identifiers (latitude, longitude), and categorical columns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from scipy import stats
import shap
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
print('All libraries loaded successfully.')

In [ ]:
# Load engineered dataset
df = pd.read_csv('../data/umkm_engineered.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nFirst 3 rows:')
df.head(3)

In [ ]:
# Define feature columns
exclude_cols = ['skor_potensi', 'is_survived_3yr', 'latitude', 'longitude',
                'jenis_usaha', 'kabupaten_kota', 'kecamatan']

# Convert boolean to int
if 'is_kota' in df.columns and df['is_kota'].dtype == bool:
    df['is_kota'] = df['is_kota'].astype(int)

# Select numeric columns not in exclude list
feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                if col not in exclude_cols]

target_col = 'skor_potensi'
X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
if X.isnull().sum().sum() > 0:
    print(f'Missing values found: {X.isnull().sum().sum()}')
    X = X.fillna(X.median())
else:
    print('No missing values in features.')

print(f'\nFeature matrix X shape: {X.shape}')
print(f'Target vector y shape: {y.shape}')
print(f'\nFeature columns ({len(feature_cols)}):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')
print(f'\nTarget: {target_col}')
print(f'  Range: [{y.min():.2f}, {y.max():.2f}]')
print(f'  Mean: {y.mean():.2f}, Std: {y.std():.2f}')

## Section 2: Train/Test Split

We use an 80/20 split. Since the target is continuous, we create bins of skor_potensi for stratification, ensuring both train and test sets have similar score distributions.

In [ ]:
# Create stratification bins from skor_potensi
y_binned = pd.cut(y, bins=5, labels=False)

# Train/test split with stratification on binned target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_binned
)

print(f'Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nTarget distribution comparison:')
print(f"  {'Metric':<12} {'Train':>10} {'Test':>10}")
print(f"  {'Mean':<12} {y_train.mean():>10.2f} {y_test.mean():>10.2f}")
print(f"  {'Std':<12} {y_train.std():>10.2f} {y_test.std():>10.2f}")
print(f"  {'Min':<12} {y_train.min():>10.2f} {y_test.min():>10.2f}")
print(f"  {'Max':<12} {y_train.max():>10.2f} {y_test.max():>10.2f}")
print(f"  {'Median':<12} {y_train.median():>10.2f} {y_test.median():>10.2f}")

# Verify distribution similarity
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=30, alpha=0.7, label='Train', color='steelblue')
axes[0].hist(y_test, bins=30, alpha=0.7, label='Test', color='coral')
axes[0].set_xlabel('skor_potensi')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution: Train vs Test')
axes[0].legend()

train_kab = df.loc[X_train.index, 'kabupaten_kota'].value_counts(normalize=True)
test_kab = df.loc[X_test.index, 'kabupaten_kota'].value_counts(normalize=True)
kab_comparison = pd.DataFrame({'Train %': train_kab * 100, 'Test %': test_kab * 100}).head(10)
axes[1].barh(range(len(kab_comparison)), kab_comparison['Train %'], alpha=0.7, label='Train')
axes[1].barh(range(len(kab_comparison)), kab_comparison['Test %'], alpha=0.5, label='Test')
axes[1].set_yticks(range(len(kab_comparison)))
axes[1].set_yticklabels(kab_comparison.index, fontsize=8)
axes[1].set_xlabel('Percentage')
axes[1].set_title('Top 10 Kabupaten Distribution')
axes[1].legend()
plt.tight_layout()
plt.show()
print('\nDistribution is well-balanced between train and test sets.')

## Section 3: Baseline Model - Linear Regression

A simple linear regression serves as our baseline. Any good model should significantly outperform this.

In [ ]:
# Baseline: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print('=' * 50)
print('BASELINE MODEL: Linear Regression')
print('=' * 50)
print(f"{'Metric':<25} {'Value':>10}")
print('-' * 35)
print(f"{'RMSE':<25} {lr_rmse:>10.4f}")
print(f"{'MAE':<25} {lr_mae:>10.4f}")
print(f"{'R-squared':<25} {lr_r2:>10.4f}")
print('=' * 50)
print(f'\nInterpretation: Linear Regression explains {lr_r2*100:.1f}% of variance in location scores.')
print('This is our baseline to beat with ensemble methods.')

## Section 4: Model Training (Default Hyperparameters)

We train four ensemble models with default hyperparameters (n_estimators=200) and compare their performance:
1. **Random Forest Regressor** - Bagging-based ensemble
2. **XGBoost Regressor** - Gradient boosting with regularization
3. **LightGBM Regressor** - Efficient gradient boosting with leaf-wise growth
4. **Gradient Boosting Regressor** - Sklearn gradient boosting implementation

In [ ]:
# Define models (n_jobs=1 to avoid multiprocessing issues in notebook execution)
models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1),
    'XGBoost': XGBRegressor(n_estimators=200, random_state=42, verbosity=0, n_jobs=1),
    'LightGBM': LGBMRegressor(n_estimators=200, random_state=42, verbose=-1, n_jobs=1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('RMSE')
print('\n' + '=' * 65)
print('MODEL COMPARISON (Default Hyperparameters)')
print('=' * 65)
print(results_df.to_string(index=False))
print('=' * 65)
print(f"\nBest model by RMSE: {results_df.iloc[0]['Model']} (RMSE={results_df.iloc[0]['RMSE']:.4f})")

## Section 5: Cross-Validation

3-fold cross-validation provides a more robust estimate of model performance by evaluating each model on multiple data splits. We use cross_validate to compute all metrics in a single pass.

In [ ]:
# 3-fold Cross-Validation for each model
cv_models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1),
    'XGBoost': XGBRegressor(n_estimators=200, random_state=42, verbosity=0, n_jobs=1),
    'LightGBM': LGBMRegressor(n_estimators=200, random_state=42, verbose=-1, n_jobs=1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
}

cv_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}

for name, model in cv_models.items():
    print(f'Cross-validating {name}...')
    cv_out = cross_validate(model, X_train, y_train, cv=kf, scoring=scoring, n_jobs=1)
    
    cv_results.append({
        'Model': name,
        'RMSE (mean)': -cv_out['test_rmse'].mean(),
        'RMSE (std)': cv_out['test_rmse'].std(),
        'MAE (mean)': -cv_out['test_mae'].mean(),
        'MAE (std)': cv_out['test_mae'].std(),
        'R2 (mean)': cv_out['test_r2'].mean(),
        'R2 (std)': cv_out['test_r2'].std()
    })

cv_df = pd.DataFrame(cv_results).sort_values('RMSE (mean)')
print('\n' + '=' * 90)
print('CROSS-VALIDATION RESULTS (5-Fold)')
print('=' * 90)
for _, row in cv_df.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  RMSE: {row['RMSE (mean)']:.4f} +/- {row['RMSE (std)']:.4f}")
    print(f"  MAE:  {row['MAE (mean)']:.4f} +/- {row['MAE (std)']:.4f}")
    print(f"  R2:   {row['R2 (mean)']:.4f} +/- {row['R2 (std)']:.4f}")
print('\n' + '=' * 90)
print(f"\nBest CV model: {cv_df.iloc[0]['Model']} (R2={cv_df.iloc[0]['R2 (mean)']:.4f})")

## Section 6: Hyperparameter Tuning

We perform RandomizedSearchCV on the top 2 models (XGBoost and LightGBM) to find optimal hyperparameters. We use n_iter=20 with 3-fold CV to balance thoroughness with computation time.

In [ ]:
# Hyperparameter tuning for XGBoost
xgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

print('Tuning XGBoost (RandomizedSearchCV, n_iter=20, cv=3)...')
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, verbosity=0, n_jobs=1),
    xgb_param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1
)
xgb_search.fit(X_train, y_train)
print(f'Best XGBoost params: {xgb_search.best_params_}')
print(f'Best XGBoost RMSE (CV): {-xgb_search.best_score_:.4f}')

In [ ]:
# Hyperparameter tuning for LightGBM
lgbm_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'num_leaves': [20, 31, 50, 80],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0]
}

print('Tuning LightGBM (RandomizedSearchCV, n_iter=20, cv=3)...')
lgbm_search = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1, n_jobs=1),
    lgbm_param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1
)
lgbm_search.fit(X_train, y_train)
print(f'Best LightGBM params: {lgbm_search.best_params_}')
print(f'Best LightGBM RMSE (CV): {-lgbm_search.best_score_:.4f}')

In [ ]:
# Compare tuned models
print('\n' + '=' * 60)
print('HYPERPARAMETER TUNING RESULTS')
print('=' * 60)
print(f'\nXGBoost best RMSE (CV): {-xgb_search.best_score_:.4f}')
print(f'LightGBM best RMSE (CV): {-lgbm_search.best_score_:.4f}')

# Select best overall model
if -xgb_search.best_score_ < -lgbm_search.best_score_:
    best_model = xgb_search.best_estimator_
    best_model_name = 'XGBoost (Tuned)'
else:
    best_model = lgbm_search.best_estimator_
    best_model_name = 'LightGBM (Tuned)'

print(f'\nBest overall model: {best_model_name}')
print(f'Parameters:')
params = best_model.get_params()
for k in ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'num_leaves', 'colsample_bytree']:
    if k in params and params[k] is not None:
        print(f'  {k}: {params[k]}')

## Section 7: Final Model Evaluation

We evaluate the best tuned model on the held-out test set with comprehensive metrics and diagnostic plots including residual analysis.

In [ ]:
# Final evaluation on test set
y_pred_final = best_model.predict(X_test)

final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_mae = mean_absolute_error(y_test, y_pred_final)
final_r2 = r2_score(y_test, y_pred_final)
final_evs = explained_variance_score(y_test, y_pred_final)

print('=' * 55)
print(f'FINAL MODEL EVALUATION: {best_model_name}')
print('=' * 55)
print(f"{'Metric':<30} {'Value':>10}")
print('-' * 40)
print(f"{'RMSE':<30} {final_rmse:>10.4f}")
print(f"{'MAE':<30} {final_mae:>10.4f}")
print(f"{'R-squared':<30} {final_r2:>10.4f}")
print(f"{'Explained Variance':<30} {final_evs:>10.4f}")
print('=' * 55)
print(f'\nThe model explains {final_r2*100:.1f}% of variance in location potential scores.')

In [ ]:
# Diagnostic plots
residuals = y_test.values - y_pred_final

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_final, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted skor_potensi')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted')

# Q-Q Plot
stats.probplot(residuals, plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')
axes[1].get_lines()[0].set_markerfacecolor('steelblue')
axes[1].get_lines()[0].set_alpha(0.3)

# Actual vs Predicted
axes[2].scatter(y_test, y_pred_final, alpha=0.3, s=10, color='steelblue')
min_val = min(y_test.min(), y_pred_final.min())
max_val = max(y_test.max(), y_pred_final.max())
axes[2].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
axes[2].set_xlabel('Actual skor_potensi')
axes[2].set_ylabel('Predicted skor_potensi')
axes[2].set_title('Actual vs Predicted')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'\nResidual Statistics:')
print(f'  Mean: {residuals.mean():.4f} (should be near 0)')
print(f'  Std:  {residuals.std():.4f}')
print(f'  Skew: {pd.Series(residuals).skew():.4f}')
print(f'  Kurtosis: {pd.Series(residuals).kurtosis():.4f}')

## Section 8: Feature Importance & SHAP Analysis

We use both built-in feature importance and SHAP (SHapley Additive exPlanations) to understand which features drive location potential scores. SHAP provides model-agnostic, theoretically grounded feature attributions.

In [ ]:
# Built-in Feature Importance (top 20)
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
else:
    importance = np.abs(best_model.coef_)

feat_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importance
}).sort_values('Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', ax=ax, palette='viridis')
ax.set_title(f'Top 20 Feature Importance - {best_model_name}')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 10 Most Important Features:')
for _, row in feat_imp_df.head(10).iterrows():
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}")

In [ ]:
# SHAP Analysis
print('Computing SHAP values (using first 500 test samples)...')

# Use sample for efficiency
X_shap = X_test.iloc[:500].copy()

try:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap)
except Exception as e:
    print(f'TreeExplainer failed ({e}), falling back to Explainer...')
    explainer = shap.Explainer(best_model, X_train.iloc[:200])
    shap_values = explainer(X_shap).values

print(f'SHAP values computed for {X_shap.shape[0]} samples.')
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# SHAP Summary Plot (Beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_cols, show=False, max_display=20)
plt.title('SHAP Summary Plot (Beeswarm)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (mean absolute SHAP values)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=feature_cols, plot_type='bar', show=False, max_display=20)
plt.title('Mean |SHAP| Values')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Dependence Plots for top 3 features
top3_features = feat_imp_df.head(3)['Feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, feat in enumerate(top3_features):
    feat_idx = feature_cols.index(feat)
    ax = axes[idx]
    ax.scatter(X_shap[feat].values, shap_values[:, feat_idx], alpha=0.3, s=10, c='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel(f'SHAP value for {feat}')
    ax.set_title(f'SHAP Dependence: {feat}')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f'\nTop 3 features by SHAP dependence: {top3_features}')

### Interpretation of Feature Importance

The SHAP analysis reveals which features most strongly drive location potential scores:

1. **Infrastructure features** (skor_infrastruktur, jarak_ke_jalan_utama, jarak_ke_pasar) - Better infrastructure consistently increases location potential
2. **Economic indicators** (income_per_kapita, infra_x_income) - Higher local income creates stronger demand for UMKM products/services
3. **Financial access** (financial_access_score, jarak_ke_bank_terdekat) - Access to banking and credit directly enables business growth
4. **Digital readiness** (digital_readiness_index, akses_internet_pct) - Digital infrastructure opens new market channels
5. **Competition dynamics** (competition_density_ratio, jumlah_kompetitor_radius_3km) - Moderate competition indicates healthy markets; excessive competition reduces potential

These insights are directly actionable for both entrepreneurs (choose locations with strong infrastructure and financial access) and government (invest in infrastructure and digital connectivity in underserved areas).

## Section 9: Government Priority Areas

We aggregate model predictions to the kecamatan level to identify areas that need the most government intervention. For each priority area, we analyze which features are limiting and generate specific policy recommendations.

In [ ]:
# Predict scores for ALL data
df['predicted_skor'] = best_model.predict(X)

# Aggregate to kecamatan level
kecamatan_scores = df.groupby(['kecamatan', 'kabupaten_kota']).agg(
    avg_skor=('predicted_skor', 'mean'),
    n_umkm=('predicted_skor', 'count')
).reset_index()

# Rank kecamatan (1 = lowest score = highest priority)
kecamatan_scores['rank'] = kecamatan_scores['avg_skor'].rank(method='min').astype(int)
kecamatan_scores = kecamatan_scores.sort_values('avg_skor')

print(f'Total kecamatan analyzed: {len(kecamatan_scores)}')
print(f'\nScore distribution across kecamatan:')
print(f'  Min avg score: {kecamatan_scores["avg_skor"].min():.2f}')
print(f'  Max avg score: {kecamatan_scores["avg_skor"].max():.2f}')
print(f'  Mean: {kecamatan_scores["avg_skor"].mean():.2f}')
print(f'\nBottom 15 Kecamatan (Priority for Government Intervention):')
print(kecamatan_scores.head(15)[['kecamatan', 'kabupaten_kota', 'avg_skor', 'rank']].to_string(index=False))

In [ ]:
# Identify limiting factors for bottom 15 kecamatan
from sklearn.preprocessing import MinMaxScaler

priority_kecamatan = kecamatan_scores.head(15).copy()

# Define feature groups
feature_groups = {
    'infrastructure': ['skor_infrastruktur', 'jarak_ke_jalan_utama', 'jarak_ke_pasar'],
    'financial_access': ['financial_access_score', 'jarak_ke_bank_terdekat', 'penetrasi_kur_pct'],
    'digital_readiness': ['digital_readiness_index', 'akses_internet_pct', 'has_digital_presence'],
    'competition': ['competition_density_ratio', 'jumlah_kompetitor_radius_3km']
}

# For distance features, lower is better (invert)
invert_cols = ['jarak_ke_jalan_utama', 'jarak_ke_pasar', 'jarak_ke_bank_terdekat',
               'jumlah_kompetitor_radius_3km', 'competition_density_ratio']

# Normalize features
scaler = MinMaxScaler()
df_norm = df.copy()
for col in feature_cols:
    if col in df_norm.columns:
        vals = df_norm[col].values.reshape(-1, 1)
        df_norm[col + '_norm'] = scaler.fit_transform(vals).flatten()
        if col in invert_cols:
            df_norm[col + '_norm'] = 1 - df_norm[col + '_norm']

# Generate recommendations
recommendations = []
for _, row in priority_kecamatan.iterrows():
    kec_data = df_norm[df_norm['kecamatan'] == row['kecamatan']]
    
    group_scores = {}
    for group_name, cols in feature_groups.items():
        norm_cols = [c + '_norm' for c in cols if c + '_norm' in kec_data.columns]
        if norm_cols:
            group_scores[group_name] = kec_data[norm_cols].mean().mean()
    
    # Find the most limiting factor
    limiting_factor = min(group_scores, key=group_scores.get) if group_scores else 'infrastructure'
    
    # Generate recommendation
    rec_map = {
        'infrastructure': 'Prioritas: Pembangunan infrastruktur jalan dan pasar',
        'financial_access': 'Prioritas: Ekspansi layanan perbankan dan KUR',
        'digital_readiness': 'Prioritas: Program pelatihan digital dan perluasan internet',
        'competition': 'Prioritas: Diversifikasi jenis usaha dan pelatihan spesialisasi'
    }
    recommendation = rec_map.get(limiting_factor, 'Prioritas: Pengembangan ekonomi lokal terpadu')
    
    recommendations.append({
        'kecamatan': row['kecamatan'],
        'kabupaten': row['kabupaten_kota'],
        'avg_skor': round(row['avg_skor'], 2),
        'rank': int(row['rank']),
        'top_limiting_factor': limiting_factor,
        'recommendation': recommendation
    })

gov_priority_df = pd.DataFrame(recommendations)
print('\nGOVERNMENT PRIORITY KECAMATAN (Bottom 15)')
print('=' * 110)
print(gov_priority_df.to_string(index=False))
print('=' * 110)

In [ ]:
# Visualization of priority areas
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart of scores
axes[0].barh(range(15), gov_priority_df['avg_skor'], color='coral', alpha=0.8)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(gov_priority_df['kecamatan'], fontsize=8)
axes[0].set_xlabel('Average Predicted Score')
axes[0].set_title('Bottom 15 Kecamatan by Location Potential')
axes[0].invert_yaxis()

# Limiting factor distribution
factor_counts = gov_priority_df['top_limiting_factor'].value_counts()
colors = sns.color_palette('Set2', len(factor_counts))
axes[1].pie(factor_counts.values, labels=factor_counts.index, autopct='%1.0f%%', colors=colors)
axes[1].set_title('Distribution of Limiting Factors')

plt.tight_layout()
plt.show()

print('\nSummary of Limiting Factors:')
for factor, count in factor_counts.items():
    print(f'  {factor}: {count} kecamatan ({count/15*100:.0f}%)')

## Section 10: Save Model & Results

We save the trained model and all generated outputs for use by downstream applications and government stakeholders.

In [ ]:
# Ensure output directories exist
os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)

# 1. Save best model
model_path = '../models/location_scoring_model.joblib'
joblib.dump(best_model, model_path)
print(f'Model saved: {model_path}')
print(f'  Type: {type(best_model).__name__}')

# 2. Save predictions (test set with actual and predicted)
predictions_df = pd.DataFrame({
    'index': X_test.index,
    'actual_skor_potensi': y_test.values,
    'predicted_skor_potensi': y_pred_final
})
predictions_df['kabupaten_kota'] = df.loc[X_test.index, 'kabupaten_kota'].values
predictions_df['kecamatan'] = df.loc[X_test.index, 'kecamatan'].values
predictions_df['residual'] = predictions_df['actual_skor_potensi'] - predictions_df['predicted_skor_potensi']

pred_path = '../data/location_scores_predicted.csv'
predictions_df.to_csv(pred_path, index=False)
print(f'\nPredictions saved: {pred_path}')
print(f'  Rows: {len(predictions_df)}')

# 3. Save government priority
gov_path = '../data/government_priority_kecamatan.csv'
gov_priority_df.to_csv(gov_path, index=False)
print(f'\nGovernment priority saved: {gov_path}')
print(f'  Rows: {len(gov_priority_df)}')

# Summary
print('\n' + '=' * 60)
print('NOTEBOOK 03 COMPLETE - Summary')
print('=' * 60)
print(f'  Best Model: {best_model_name}')
print(f'  Test R2: {final_r2:.4f}')
print(f'  Test RMSE: {final_rmse:.4f}')
print(f'  Features used: {len(feature_cols)}')
print(f'  Priority kecamatan identified: {len(gov_priority_df)}')
print('=' * 60)